# Character Introduction Analysis

Analysis of 1,401 character introductions across 102 texts, annotated by CharacterIntroTask.
Data: `~/Dropbox/Prof/Books/AbsLitHist/data/llm_annotations/character_intros.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

df = pd.read_csv(os.path.expanduser('~/Dropbox/Prof/Books/AbsLitHist/data/llm_annotations/character_intros.csv'))
df['period'] = (df['meta_year'] // 50 * 50).astype('Int64')

outdir = os.path.expanduser('~/Dropbox/Prof/Books/AbsLitHist/book/figures')

print(f"{len(df)} character intros across {df['meta__id'].nunique()} texts")
print(f"Years: {df['meta_year'].dropna().min():.0f}\u2013{df['meta_year'].dropna().max():.0f}")

## Historical arc

In [ ]:
# Intro mode by half-century
print("=== Intro mode by half-century ===")
ct = pd.crosstab(df['period'], df['intro_mode'], normalize='index').round(3)
period_n = df['period'].value_counts().sort_index()
for p in ct.index:
    n = period_n.get(p, 0)
    if n < 20:
        continue
    abstract = ct.loc[p, 'abstract_social'] + ct.loc[p, 'abstract_moral']
    physical = ct.loc[p, 'physical_appearance'] if 'physical_appearance' in ct.columns else 0
    behav = ct.loc[p, 'behavioral'] if 'behavioral' in ct.columns else 0
    print(f"  {p}: n={n:>4}  abstract={abstract:.0%}  physical={physical:.0%}  behavioral={behav:.0%}")

# Descriptor register by half-century
print("\n=== Descriptor register by half-century ===")
ct2 = pd.crosstab(df['period'], df['descriptor_register'], normalize='index').round(3)
for p in ct2.index:
    n = period_n.get(p, 0)
    if n < 20:
        continue
    abst = ct2.loc[p, 'predominantly_abstract'] if 'predominantly_abstract' in ct2.columns else 0
    conc = ct2.loc[p, 'predominantly_concrete'] if 'predominantly_concrete' in ct2.columns else 0
    bal = ct2.loc[p, 'balanced'] if 'balanced' in ct2.columns else 0
    print(f"  {p}: n={n:>4}  abstract={abst:.0%}  concrete={conc:.0%}  balanced={bal:.0%}")

In [ ]:
# Figure 1: Historical arc
pn = df['period'].value_counts().sort_index()
periods = [p for p in sorted(pn.index) if pn[p] >= 20]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1a: Intro mode over time
ct = pd.crosstab(df['period'], df['intro_mode'], normalize='index')
for mode, color, label in [
    ('abstract_social', '#d62728', 'Abstract social'),
    ('abstract_moral', '#ff7f0e', 'Abstract moral'),
    ('physical_appearance', '#2ca02c', 'Physical appearance'),
    ('behavioral', '#1f77b4', 'Behavioral'),
    ('relational', '#9467bd', 'Relational'),
]:
    if mode in ct.columns:
        vals = ct.loc[periods, mode]
        axes[0].plot(vals.index, vals.values, 'o-', color=color, label=label, markersize=5)
axes[0].set_title('Character introduction mode over time')
axes[0].set_xlabel('Half-century')
axes[0].set_ylabel('Proportion of characters')
axes[0].legend(fontsize=8)
axes[0].set_xlim(1530, 2030)

# 1b: Descriptor register over time
ct2 = pd.crosstab(df['period'], df['descriptor_register'], normalize='index')
for reg, color, label in [
    ('predominantly_abstract', '#d62728', 'Predominantly abstract'),
    ('balanced', '#1f77b4', 'Balanced'),
    ('predominantly_concrete', '#2ca02c', 'Predominantly concrete'),
]:
    if reg in ct2.columns:
        vals = ct2.loc[periods, reg]
        axes[1].plot(vals.index, vals.values, 'o-', color=color, label=label, markersize=5)
axes[1].set_title('Descriptor register over time')
axes[1].set_xlabel('Half-century')
axes[1].set_ylabel('Proportion')
axes[1].legend(fontsize=8)
axes[1].set_xlim(1530, 2030)

# 1c: Social legibility + narrator assessment over time
ct3 = pd.crosstab(df['period'], df['social_legibility'], normalize='index')
ct4 = pd.crosstab(df['period'], df['is_narrator_assessment'], normalize='index')
if 'transparent' in ct3.columns:
    vals = ct3.loc[periods, 'transparent']
    axes[2].plot(vals.index, vals.values, 'o-', color='#1f77b4', label='Transparent (legible)', markersize=5)
if 'opaque' in ct3.columns:
    vals = ct3.loc[periods, 'opaque']
    axes[2].plot(vals.index, vals.values, 'o-', color='#d62728', label='Opaque', markersize=5)
if True in ct4.columns:
    vals = ct4.loc[periods, True]
    axes[2].plot(vals.index, vals.values, 's--', color='#ff7f0e', label='Narrator assessment (telling)', markersize=5)
axes[2].set_title('Legibility and narration over time')
axes[2].set_xlabel('Half-century')
axes[2].set_ylabel('Proportion')
axes[2].legend(fontsize=8)
axes[2].set_xlim(1530, 2030)

plt.tight_layout()
plt.savefig(os.path.join(outdir, 'char_intro_historical_arc.png'), dpi=150, bbox_inches='tight')
plt.show()

## Class gradient

In [ ]:
# Class x intro mode
print("=== Intro mode by social class ===")
ct = pd.crosstab(df['social_class'], df['intro_mode'], normalize='index').round(3)
counts = df['social_class'].value_counts()
keep = counts[counts >= 15].index
for cls in keep:
    abstract = ct.loc[cls, 'abstract_social'] + ct.loc[cls, 'abstract_moral']
    physical = ct.loc[cls].get('physical_appearance', 0)
    behav = ct.loc[cls].get('behavioral', 0)
    print(f"  {cls:<25s} n={counts[cls]:>4}  abstract={abstract:.0%}  physical={physical:.0%}  behavioral={behav:.0%}")

# Class x register
print("\n=== Descriptor register by social class ===")
ct2 = pd.crosstab(df['social_class'], df['descriptor_register'], normalize='index').round(3)
for cls in keep:
    abst = ct2.loc[cls].get('predominantly_abstract', 0)
    conc = ct2.loc[cls].get('predominantly_concrete', 0)
    print(f"  {cls:<25s} n={counts[cls]:>4}  abstract={abst:.0%}  concrete={conc:.0%}")

# Class composition over time
print("\n=== Class composition by half-century ===")
ct3 = pd.crosstab(df['period'], df['social_class'], normalize='index').round(3)
for p in sorted(pn.index):
    if pn[p] < 20:
        continue
    aris = ct3.loc[p].get('royalty_aristocracy', 0)
    gent = ct3.loc[p].get('gentry', 0)
    mid = ct3.loc[p].get('middling', 0)
    serv = ct3.loc[p].get('servant_laborer', 0)
    print(f"  {p}: n={pn[p]:>4}  aristocracy={aris:.0%}  gentry={gent:.0%}  middling={mid:.0%}  servant={serv:.0%}")

In [ ]:
# Figure 2: Class gradient
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_order = ['royalty_aristocracy', 'gentry', 'professional', 'middling', 'servant_laborer']
class_labels = ['Royalty/\naristocracy', 'Gentry', 'Professional', 'Middling', 'Servant/\nlaborer']
class_n = df['social_class'].value_counts()

# 2a: Abstract intro rate by class
ct_cls = pd.crosstab(df['social_class'], df['intro_mode'], normalize='index')
abstract_rates = []
for cls in class_order:
    if cls in ct_cls.index:
        r = ct_cls.loc[cls].get('abstract_social', 0) + ct_cls.loc[cls].get('abstract_moral', 0)
        abstract_rates.append(r)
    else:
        abstract_rates.append(0)
bars = axes[0].bar(range(len(class_order)), abstract_rates, color=['#d62728','#ff7f0e','#1f77b4','#2ca02c','#9467bd'])
axes[0].set_xticks(range(len(class_order)))
axes[0].set_xticklabels(class_labels, fontsize=9)
axes[0].set_ylabel('Abstract introduction rate')
axes[0].set_title('Abstract character introduction by social class')
for i, (r, cls) in enumerate(zip(abstract_rates, class_order)):
    n = class_n.get(cls, 0)
    axes[0].text(i, r + 0.01, f'n={n}', ha='center', fontsize=8)

# 2b: Register by class (stacked bar)
ct_reg = pd.crosstab(df['social_class'], df['descriptor_register'], normalize='index')
bottoms = np.zeros(len(class_order))
for reg, color, label in [
    ('predominantly_abstract', '#d62728', 'Abstract'),
    ('abstract_punctuated', '#ff7f0e', 'Abstract punctuated'),
    ('balanced', '#aaaaaa', 'Balanced'),
    ('concrete_punctuated', '#7fbf7f', 'Concrete punctuated'),
    ('predominantly_concrete', '#2ca02c', 'Concrete'),
]:
    if reg in ct_reg.columns:
        vals = [ct_reg.loc[cls, reg] if cls in ct_reg.index else 0 for cls in class_order]
        axes[1].bar(range(len(class_order)), vals, bottom=bottoms, color=color, label=label)
        bottoms += np.array(vals)
axes[1].set_xticks(range(len(class_order)))
axes[1].set_xticklabels(class_labels, fontsize=9)
axes[1].set_ylabel('Proportion')
axes[1].set_title('Descriptor register by social class')
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(outdir, 'char_intro_class_gradient.png'), dpi=150, bbox_inches='tight')
plt.show()

## Character role

In [ ]:
# Role x intro mode
print("=== Character role x intro mode ===")
ct = pd.crosstab(df['character_role'], df['intro_mode'], normalize='index').round(3)
counts = df['character_role'].value_counts()
for role in counts.index:
    if counts[role] < 10:
        continue
    abs_s = ct.loc[role].get('abstract_social', 0) + ct.loc[role].get('abstract_moral', 0)
    phys = ct.loc[role].get('physical_appearance', 0)
    beh = ct.loc[role].get('behavioral', 0)
    rel = ct.loc[role].get('relational', 0)
    mat = ct.loc[role].get('material_commodity', 0)
    print(f"  {role:<20s} n={counts[role]:>4}  abstract={abs_s:.0%}  physical={phys:.0%}  behavioral={beh:.0%}  relational={rel:.0%}  material={mat:.0%}")

# Role x register
print("\n=== Character role x descriptor register ===")
ct2 = pd.crosstab(df['character_role'], df['descriptor_register'], normalize='index').round(3)
for role in counts.index:
    if counts[role] < 10:
        continue
    abst = ct2.loc[role].get('predominantly_abstract', 0)
    conc = ct2.loc[role].get('predominantly_concrete', 0)
    bal = ct2.loc[role].get('balanced', 0)
    print(f"  {role:<20s} n={counts[role]:>4}  abstract={abst:.0%}  concrete={conc:.0%}  balanced={bal:.0%}")

In [ ]:
# Figure 3: Role patterns
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

role_order = ['protagonist', 'love_interest', 'antagonist', 'authority_figure', 'confidant', 'minor']
role_labels = ['Protagonist', 'Love interest', 'Antagonist', 'Authority', 'Confidant', 'Minor']
ct_role = pd.crosstab(df['character_role'], df['intro_mode'], normalize='index')
role_n = df['character_role'].value_counts()

x = np.arange(len(role_order))
width = 0.15
for i, (mode, color, label) in enumerate([
    ('abstract_social', '#d62728', 'Abstract social'),
    ('abstract_moral', '#ff7f0e', 'Abstract moral'),
    ('physical_appearance', '#2ca02c', 'Physical'),
    ('behavioral', '#1f77b4', 'Behavioral'),
    ('relational', '#9467bd', 'Relational'),
]):
    vals = [ct_role.loc[r, mode] if r in ct_role.index and mode in ct_role.columns else 0 for r in role_order]
    ax.bar(x + i * width, vals, width, color=color, label=label)

ax.set_xticks(x + width * 2)
ax.set_xticklabels([f'{l}\n(n={role_n.get(r,0)})' for l, r in zip(role_labels, role_order)], fontsize=9)
ax.set_ylabel('Proportion')
ax.set_title('Character introduction mode by narrative role')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(outdir, 'char_intro_by_role.png'), dpi=150, bbox_inches='tight')
plt.show()

## Gender, interiority, legibility, narration

## Shift-share decomposition

Oaxaca-Blinder decomposition of the change in passage-level abstractness between periods. For each factor (social class, intro mode, etc.), decomposes total change into:
- **Composition**: how much change is due to the *mix* of groups shifting (e.g. fewer aristocrats)
- **Within**: how much is due to *within-group* means changing (e.g. gentry descriptions get more concrete)
- **Interaction**: correlated changes in both

In [ ]:
df_scored = pd.read_csv(os.path.expanduser('~/Dropbox/Prof/Books/AbsLitHist/data/llm_annotations/character_intros_scored.csv'))
df_scored = df_scored.dropna(subset=['passage_score', 'meta_year'])

def year_to_period(y):
    if y < 1700: return 'pre-1700'
    elif y < 1800: return 'C18'
    else: return 'post-1800'

df_scored['period'] = df_scored['meta_year'].apply(year_to_period)

def shift_share(df, period_a, period_b, group_col, outcome_col='passage_score'):
    a = df[df['period'] == period_a]
    b = df[df['period'] == period_b]
    groups = sorted(set(a[group_col].dropna()) | set(b[group_col].dropna()))
    share_a = a[group_col].value_counts(normalize=True)
    share_b = b[group_col].value_counts(normalize=True)
    mean_a = a.groupby(group_col)[outcome_col].mean()
    mean_b = b.groupby(group_col)[outcome_col].mean()
    overall_a, overall_b = a[outcome_col].mean(), b[outcome_col].mean()
    tc = overall_b - overall_a
    comp, with_, inter = 0, 0, 0
    details = []
    for g in groups:
        sa, sb = share_a.get(g, 0), share_b.get(g, 0)
        ma, mb = mean_a.get(g, overall_a), mean_b.get(g, overall_b)
        ds, dm = sb - sa, mb - ma
        c, w, ix = ds * ma, sa * dm, ds * dm
        comp += c; with_ += w; inter += ix
        details.append({'group': g, 'share_a': sa, 'share_b': sb, 'delta_share': ds,
                        'mean_a': ma, 'mean_b': mb, 'delta_mean': dm,
                        'composition': c, 'within': w, 'interaction': ix})
    return {'period_a': period_a, 'period_b': period_b,
            'mean_a': overall_a, 'mean_b': overall_b, 'total_change': tc,
            'composition': comp, 'within': with_, 'interaction': inter,
            'details': pd.DataFrame(details)}

factors = {
    'social_class': 'Social class',
    'intro_mode': 'Introduction mode',
    'descriptor_register': 'Descriptor register',
    'character_role': 'Character role',
    'social_legibility': 'Social legibility',
}

for pa, pb in [('pre-1700', 'C18'), ('C18', 'post-1800'), ('pre-1700', 'post-1800')]:
    na = len(df_scored[df_scored.period == pa])
    nb = len(df_scored[df_scored.period == pb])
    ma = df_scored[df_scored.period == pa].passage_score.mean()
    mb = df_scored[df_scored.period == pb].passage_score.mean()
    print(f"\n{'='*70}")
    print(f"{pa} (n={na}, mean={ma:+.3f}) -> {pb} (n={nb}, mean={mb:+.3f})  change={mb-ma:+.3f}")
    print(f"{'='*70}")
    for gcol, label in factors.items():
        r = shift_share(df_scored, pa, pb, gcol)
        tc = r['total_change']
        if abs(tc) < 0.001: continue
        print(f"\n  {label}:")
        print(f"    Composition: {r['composition']:+.4f} ({r['composition']/tc*100:+.0f}%)")
        print(f"    Within:      {r['within']:+.4f} ({r['within']/tc*100:+.0f}%)")
        print(f"    Interaction: {r['interaction']:+.4f} ({r['interaction']/tc*100:+.0f}%)")
        det = r['details']
        det['total'] = det['composition'] + det['within'] + det['interaction']
        for _, d in det.sort_values('total', key=abs, ascending=False).head(3).iterrows():
            if abs(d['total']) < 0.001: continue
            print(f"      {d['group']:<25s} share {d['share_a']:.2f}->{d['share_b']:.2f}  "
                  f"mean {d['mean_a']:+.3f}->{d['mean_b']:+.3f}")

In [ ]:
# Gender x register
print("=== Gender x descriptor register ===")
ct = pd.crosstab(df['character_gender'], df['descriptor_register'], normalize='index').round(3)
gender_n = df['character_gender'].value_counts()
for g in ['female', 'male']:
    abst = ct.loc[g, 'predominantly_abstract']
    conc = ct.loc[g, 'predominantly_concrete']
    print(f"  {g:<10s} n={gender_n[g]:>4}  abstract={abst:.0%}  concrete={conc:.0%}")

# Interiority by period
print("\n=== Interiority by half-century ===")
ct2 = pd.crosstab(df['period'], df['interiority'], normalize='index').round(3)
for p in sorted(pn.index):
    if pn[p] < 20:
        continue
    ext = ct2.loc[p].get('external_only', 0)
    imp = ct2.loc[p].get('implied_interiority', 0)
    int_ = ct2.loc[p].get('interior_access', 0)
    print(f"  {p}: n={pn[p]:>4}  external={ext:.0%}  implied={imp:.0%}  interior={int_:.0%}")

# Describing voice by period
print("\n=== Describing voice by half-century ===")
ct3 = pd.crosstab(df['period'], df['describing_voice'], normalize='index').round(3)
for p in sorted(pn.index):
    if pn[p] < 20:
        continue
    narr = ct3.loc[p].get('narrator', 0)
    other = ct3.loc[p].get('other_character', 0)
    self_ = ct3.loc[p].get('self', 0)
    print(f"  {p}: n={pn[p]:>4}  narrator={narr:.0%}  other_char={other:.0%}  self={self_:.0%}")

# Legibility by period
print("\n=== Social legibility by half-century ===")
ct4 = pd.crosstab(df['period'], df['social_legibility'], normalize='index').round(3)
for p in sorted(pn.index):
    if pn[p] < 20:
        continue
    trans = ct4.loc[p].get('transparent', 0)
    opaq = ct4.loc[p].get('opaque', 0)
    print(f"  {p}: n={pn[p]:>4}  transparent={trans:.0%}  opaque={opaq:.0%}")

# Narrator assessment by period
print("\n=== Narrator assessment (telling) by half-century ===")
ct5 = pd.crosstab(df['period'], df['is_narrator_assessment'], normalize='index').round(3)
for p in sorted(pn.index):
    if pn[p] < 20:
        continue
    telling = ct5.loc[p].get(True, 0)
    print(f"  {p}: n={pn[p]:>4}  narrator_assessment={telling:.0%}")

In [ ]:
# Figure 4: Gender over time
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for gi, gender in enumerate(['female', 'male']):
    sub = df[df['character_gender'] == gender]
    ct_g = pd.crosstab(sub['period'], sub['descriptor_register'], normalize='index')
    gperiods = [p for p in periods if p in ct_g.index]
    for reg, color, label in [
        ('predominantly_abstract', '#d62728', 'Abstract'),
        ('balanced', '#1f77b4', 'Balanced'),
        ('predominantly_concrete', '#2ca02c', 'Concrete'),
    ]:
        if reg in ct_g.columns:
            vals = ct_g.loc[gperiods, reg]
            axes[gi].plot(vals.index, vals.values, 'o-', color=color, label=label, markersize=4)
    axes[gi].set_title(f'{gender.title()} characters (n={len(sub)})')
    axes[gi].set_xlabel('Half-century')
    axes[gi].set_ylabel('Proportion')
    axes[gi].legend(fontsize=8)
    axes[gi].set_xlim(1530, 2030)

plt.tight_layout()
plt.savefig(os.path.join(outdir, 'char_intro_by_gender.png'), dpi=150, bbox_inches='tight')
plt.show()

## Austen vs Dickens comparison

In [ ]:
for label, pattern in [('Austen', 'Austen'), ('Dickens', 'Dickens')]:
    sub = df[df['meta_author'].str.contains(pattern, na=False)]
    if len(sub) == 0:
        continue
    print(f"{'='*50}")
    print(f"{label}: {len(sub)} characters\n")
    print("Intro mode:")
    print(sub['intro_mode'].value_counts().to_string())
    print("\nDescriptor register:")
    print(sub['descriptor_register'].value_counts().to_string())
    print("\nCharacters:")
    for _, r in sub.head(10).iterrows():
        print(f"  {r['character_name']:<25s} mode={r['intro_mode']:<20s} register={r['descriptor_register']}")
    print()

## Love interests

In [ ]:
li = df[df['character_role'] == 'love_interest'].sort_values('meta_year')
print(f"{len(li)} love interests\n")
for _, r in li.iterrows():
    title = (r['meta_title'] or '')[:35]
    year = f"{r['meta_year']:.0f}" if pd.notna(r['meta_year']) else '?'
    gender = r['character_gender']
    mode = r['intro_mode']
    print(f"  {year:>4}  {r['character_name']:<25s} {gender:<7s} {mode:<22s} {title}")